In [51]:
# Load single CSV file
import pandas as pd
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

In [2]:
bkg_path = Path('/n/holystore01/LABS/iaifi_lab/Lab/kyoon/BNSReg/outputs/RESULTS/SIG_VS_BKG_TEST_20260206/bkg_only/bns_eval.csv')
sig_path = Path('/n/holystore01/LABS/iaifi_lab/Lab/kyoon/BNSReg/outputs/RESULTS/SIG_VS_BKG_TEST_20260206/injected/bns_eval.csv')

In [3]:
bkg_df = pd.read_csv(bkg_path)
sig_df = pd.read_csv(sig_path)

bkg_df['class'] = 'bkg'
sig_df['class'] = 'sig'
comb_df = pd.concat([bkg_df, sig_df])

In [4]:
comb_df['diff_chirp_mass'] = comb_df['pred_chirp_mass'] - comb_df['truth_chirp_mass']
comb_df['abs_diff_chirp_mass'] = comb_df['diff_chirp_mass'].abs()
comb_df['rel_error'] = comb_df['diff_chirp_mass'] / comb_df['truth_chirp_mass']
comb_df['abs_rel_error'] = comb_df['abs_diff_chirp_mass'] / comb_df['truth_chirp_mass']

In [5]:
comb_df

,pred_chirp_mass,truth_chirp_mass,snr,class,diff_chirp_mass,abs_diff_chirp_mass,rel_error,abs_rel_error
0,1.686607,1.201036,43.306664,bkg,0.485572,0.485572,0.404294,0.404294
1,1.407352,1.535445,24.462730,bkg,-0.128093,0.128093,-0.083424,0.083424
2,1.701834,1.837466,13.142533,bkg,-0.135632,0.135632,-0.073815,0.073815
3,1.225815,1.938861,17.057405,bkg,-0.713046,0.713046,-0.367765,0.367765
4,1.009498,1.400603,21.590193,bkg,-0.391105,0.391105,-0.279241,0.279241
...,...,...,...,...,...,...,...,...
7995,1.397590,1.396638,36.409450,sig,0.000952,0.000952,0.000682,0.000682
7996,1.293332,1.194137,21.980438,sig,0.099195,0.099195,0.083068,0.083068
7997,1.006327,1.018565,39.653866,sig,-0.012238,0.012238,-0.012014,0.012014
7998,1.192187,1.227626,13.041298,sig,-0.035440,0.035440,-0.028868,0.028868


# SNR vs. Abs diff

In [6]:
fig_abs_diff_vs_snr = px.scatter(
    comb_df,
    x='snr',
    y='abs_diff_chirp_mass',
    color='class'
)
fig_abs_diff_vs_snr.show()

In [7]:
fig_rel_error_vs_snr = px.scatter(
    comb_df,
    x='snr',
    y='rel_error',
    color='class'
)
fig_rel_error_vs_snr.show()

# Sig vs. Bkg discriminator

In [8]:
fig_abs_diff_sig_vs_bkg = px.histogram(
    comb_df.where(comb_df['snr'] >= 20),
    x='abs_diff_chirp_mass',
    color='class',
    opacity=0.8,
    nbins=100,
    range_x=[0, 1]
)
fig_abs_diff_sig_vs_bkg.show()

In [9]:
fig_rel_error_sig_vs_bkg = px.histogram(
    comb_df.where(comb_df['snr'] >= 20),
    x='abs_rel_error',
    color='class',
    opacity=0.5,
    nbins=200,
    range_x=[0,1]
)

fig_rel_error_sig_vs_bkg.show()

# Find best cut

In [10]:
df_ = comb_df.where(comb_df['snr'] >= 20).dropna()

In [11]:
best_ratio = 0. # S / sqrt(B)
best_S = 0
best_B = 0
best_cut = 0.
for i in np.linspace(0.01, 1, 100, endpoint=True):
    S = len(df_[(df_['abs_rel_error'] < i) & (df_['class'] == 'sig')])
    B = len(df_[(df_['abs_rel_error'] < i) & (df_['class'] == 'bkg')])
    if not (S !=0 and B == 0):
        ratio = S / np.sqrt(float(B))
    else:
        print(f'B = 0 at {i=} but {S=}')
        break
    if ratio > best_ratio:
        best_ratio = ratio
        best_S = S
        best_B = B
        best_cut = i
print(f'{best_cut=:.3f} | {best_ratio=}, {best_S=} (total: {(df_['class']=='sig').sum()}), {best_B=} (total: {(df_['class']=='bkg').sum()}), ')

best_cut=0.020 | best_ratio=np.float64(287.17718355763844), best_S=5201 (total: 5976), best_B=328 (total: 5976), 


In [12]:
fig_rel_error_sig_vs_bkg.add_vline(x=0.020, line_width=2, line_dash='dash', line_color='red')
fig_rel_error_sig_vs_bkg.show()

# SNR vs Detection Rate

In [43]:
bin_edges = [i for i in range(20, 51)]
df_['snr_bin'] = pd.cut(df_['snr'], bins=bin_edges)

In [47]:
snr_bins_sig_count = (
    df_.query("`class` == 'sig'")
       .groupby('snr_bin', observed=True)
       .size()
)
snr_bins_bkg_count = (
    df_.query("`class` == 'bkg'")
       .groupby('snr_bin', observed=True)
       .size()
) # It is supposed to be identical to sig
snr_bins_total_count = (
    df_.groupby('snr_bin', observed=True)
       .size()
)

In [ ]:
snr_bin_sig_pass_count = (
    df_.loc[df_['class'] == 'sig']
      .assign(_pass=lambda d: d['abs_rel_error'].lt(best_cut))
      .groupby('snr_bin', observed=True)['_pass']
      .sum()
      .astype('int64')
)
snr_bin_bkg_pass_count = (
    df_.loc[df_['class'] == 'bkg']
      .assign(_pass=lambda d: d['abs_rel_error'].lt(best_cut))
      .groupby('snr_bin', observed=True)['_pass']
      .sum()
      .astype('int64')
)

In [55]:
sig_ratio = snr_bin_sig_pass_count / snr_bins_sig_count
bkg_ratio = snr_bin_bkg_pass_count / snr_bins_bkg_count

In [63]:
# x at bin centers (20.5, 21.5, …)
sig_x = [iv.mid for iv in sig_ratio.index]
bkg_x = [iv.mid for iv in bkg_ratio.index]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=sig_x, y=sig_ratio.values,
    mode='lines+markers', name='sig'
))

fig.add_trace(go.Scatter(
    x=bkg_x, y=bkg_ratio.values,
    mode='lines+markers', name='bkg'
))

# ticks at integers, labels 20, 21, 22, …
ticks = list(range(20, 50))

fig.update_layout(
    xaxis=dict(
        title='SNR',
        tickmode='array',
        tickvals=ticks,
        ticktext=[str(t) for t in ticks]
    ),
    yaxis_title=f'Detection rate at {best_cut*100:.1f}% / SNR'
)

fig.show()
